# Análise de Qualidade de Variáveis

Este notebook realiza análise de drift mensal, trimestral e semestral das variáveis do dataset, utilizando KS e PSI para quantitativas e V de Cramer para categóricas. Para cada estatística calculada, uma célula markdown explica definição, hipóteses, intuição e interpretação. Testes estatísticos são aplicados para validar conclusões, com atenção ao impacto do tamanho da amostra na sensibilidade da hipótese nula. Resultados são organizados por variável e período, sem prints ou comentários redundantes.

## Teste KS (Kolmogorov-Smirnov)

Definição: Mede a diferença máxima entre distribuições acumuladas de uma variável em dois períodos.

Hipótese nula (H0): As distribuições são iguais.

Hipótese alternativa (Ha): São diferentes.

Interpretação: p < 0.05 = potencial drift; p > 0.05 = distribuições similares.

## Teste PSI (Population Stability Index)

Definição: Mede o grau de mudança entre distribuições de uma variável.

Hipótese nula (H0): PSI < 0.1, estabilidade.

Hipótese alternativa (Ha): PSI > 0.1, instabilidade.

Interpretação: PSI < 0.1: estável; entre 0.1–0.25: leve drift; > 0.25: drift severo.

## Teste V de Cramer (categóricas com target)

Definição: Mede associação entre uma variável categórica e o target binário.

Hipótese nula (H0): Não há associação.

Hipótese alternativa (Ha): Existe associação.

Interpretação: V próximo a zero: independente; > 0.15: associação relevante.

## Sensibilidade da hipótese nula

Testes estatísticos tendem a rejeitar a H0 em amostras grandes mesmo para pequenas diferenças. Considere, além do p-valor, a magnitude do efeito.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

df_spark = spark.table('fraud_detection_dev.gold.fraud_features')

exclude = {'TransactionDT', 'target', 'split', 'TransactionID'}
num_cols = [f.name for f in df_spark.schema.fields
            if f.dataType.typeName() in ('double', 'float', 'int', 'bigint', 'long')
            and f.name not in exclude]
cat_cols = [f.name for f in df_spark.schema.fields
            if f.dataType.typeName() == 'string' and f.name not in exclude]

period_defs = {
    'mensal': F.date_format('TransactionDT', 'yyyy-MM'),
    'trimestral': F.concat_ws('-', F.year('TransactionDT'), F.quarter('TransactionDT').cast('string')),
    'semestral': F.concat_ws('-', F.year('TransactionDT'),
               (F.floor((F.month('TransactionDT') - 1) / 6) + 1).cast('string'))
}

results = []

for period_type, period_expr in period_defs.items():
    df = df_spark.select('*', period_expr.alias('period'))
    periods = [r['period'] for r in df.select('period').distinct().orderBy('period').collect()]

    batch_size = 40
    all_hist = pd.DataFrame()
    for bi in range(0, len(num_cols), batch_size):
        batch = num_cols[bi:bi + batch_size]
        stack_args = ', '.join([f"'{c}', cast(`{c}` as double)" for c in batch])
        stacked = df.select('period',
                F.expr(f"stack({len(batch)}, {stack_args}) as (col_name, value)")) \
            .filter(F.col('value').isNotNull())

        bounds = stacked.groupBy('col_name').agg(
            F.min('value').alias('min_val'), F.max('value').alias('max_val'))

        bucketed = stacked.join(bounds, 'col_name').withColumn('bucket',
            F.when(F.col('max_val') > F.col('min_val'),
                F.least(F.lit(10), F.greatest(F.lit(1),
                    F.ceil((F.col('value') - F.col('min_val')) /
                           (F.col('max_val') - F.col('min_val')) * 10))))
             .otherwise(F.lit(1)))

        hist = bucketed.groupBy('col_name', 'period', 'bucket').count()
        all_hist = pd.concat([all_hist, hist.toPandas()])

    for col_name in num_cols:
        col_hist = all_hist[all_hist['col_name'] == col_name]
        for i in range(len(periods) - 1):
            prev_p, curr_p = periods[i], periods[i + 1]
            prev_counts = np.zeros(10)
            curr_counts = np.zeros(10)
            for _, row in col_hist[col_hist['period'] == prev_p].iterrows():
                b = int(row['bucket'])
                if 1 <= b <= 10: prev_counts[b - 1] = row['count']
            for _, row in col_hist[col_hist['period'] == curr_p].iterrows():
                b = int(row['bucket'])
                if 1 <= b <= 10: curr_counts[b - 1] = row['count']
            prev_pct = prev_counts / max(1, prev_counts.sum())
            curr_pct = curr_counts / max(1, curr_counts.sum())
            psi = float(np.sum((prev_pct - curr_pct) *
                        np.log((prev_pct + 1e-5) / (curr_pct + 1e-5))))
            ks_approx = float(np.max(np.abs(np.cumsum(prev_pct) - np.cumsum(curr_pct))))
            results.append({'period_type': period_type, 'col': col_name,
                'periods': (prev_p, curr_p), 'KS': ks_approx, 'PSI': psi})

    target_rates = df.groupBy('period').agg(F.mean('target').alias('rate')) \
        .orderBy('period').collect()
    for i in range(len(periods) - 1):
        results.append({'period_type': period_type, 'col': 'target',
            'periods': (periods[i], periods[i+1]),
            'target_prev': target_rates[i]['rate'],
            'target_curr': target_rates[i+1]['rate']})

    for col_name in cat_cols:
        for i in range(len(periods) - 1):
            prev_p, curr_p = periods[i], periods[i + 1]
            for _df_filt, label in [(df.filter(F.col('period') == prev_p), 'prev'),
                                    (df.filter(F.col('period') == curr_p), 'curr')]:
                _pd = _df_filt.select(col_name, 'target').dropna().toPandas()
                if not _pd.empty:
                    tab = pd.crosstab(_pd[col_name], _pd['target'])
                    if tab.size > 0 and tab.shape[0] > 1 and tab.shape[1] > 1:
                        from scipy.stats import chi2_contingency
                        chi2, _, _, _ = chi2_contingency(tab)
                        n = tab.sum().sum()
                        r, k = tab.shape
                        cramer_v = float(np.sqrt(chi2 / (n * (min(k-1, r-1))))) if n > 0 else None
                        results.append({'period_type': period_type, 'col': col_name,
                            'periods': (prev_p, curr_p), f'CramerV_{label}': cramer_v})

display(pd.DataFrame(results))


In [0]:
import pandas as pd

results_df = pd.DataFrame(results)

ks_df = results_df.dropna(subset=['KS']).copy()
psi_df = results_df.dropna(subset=['PSI']).copy()
target_df = results_df[results_df['col']=='target'].dropna(subset=['target_prev','target_curr']).copy()
cramer_df = results_df.dropna(subset=['CramerV_prev','CramerV_curr']).copy()

ks_summary = ks_df.groupby('col')['KS'].agg(['max','mean']).reset_index()
ks_summary.columns = ['col','KS_max','KS_mean']
ks_summary = ks_summary.sort_values('KS_max', ascending=False)

psi_summary = psi_df.groupby('col')['PSI'].agg(['max','mean']).reset_index()
psi_summary.columns = ['col','PSI_max','PSI_mean']
psi_summary = psi_summary.sort_values('PSI_max', ascending=False)

def psi_class(v):
    if v > 0.25: return 'drift_severo'
    elif v > 0.1: return 'drift_leve'
    else: return 'estavel'

psi_summary['PSI_classificacao'] = psi_summary['PSI_max'].apply(psi_class)

target_df['target_delta'] = target_df['target_curr'] - target_df['target_prev']
target_summary = target_df.groupby('period_type')['target_delta'].agg(['max','min','mean']).reset_index()

merge = ks_summary.merge(psi_summary, on='col', how='outer')

display(merge.head(30))

drift_severo = psi_summary[psi_summary['PSI_classificacao']=='drift_severo']['col'].tolist()
drift_leve = psi_summary[psi_summary['PSI_classificacao']=='drift_leve']['col'].tolist()
estavel = psi_summary[psi_summary['PSI_classificacao']=='estavel']['col'].tolist()

top_ks = ks_summary.head(10)
top_psi = psi_summary.head(10)

display(top_ks)
display(top_psi)
display(target_summary)

display(pd.DataFrame({
    'categoria': ['drift_severo (>0.25)','drift_leve (0.1-0.25)','estavel (<0.1)'],
    'n_variaveis': [len(drift_severo), len(drift_leve), len(estavel)],
    'variaveis': [', '.join(drift_severo[:10]), ', '.join(drift_leve[:10]), f'{len(estavel)} variaveis estaveis']
}))

## Recomendação estatística de retreino

### Resumo dos resultados

| Categoria | Nº variáveis | Exemplos |
| --- | --- | --- |
| Drift severo (PSI > 0.25) | 11 | id_13 (PSI=4.47), addr1_tx_count (3.53), card1_tx_count (0.73), V168, V171, V175, V209, V210, D4, id_17 |
| Drift leve (0.1 < PSI < 0.25) | 9 | D11 (0.25), D2 (0.20), V237, V236, D10 (0.15), id_20, id_02, V169, V217 |
| Estável (PSI < 0.1) | 100 | TransactionAmt, C4, C5, D3, D5, V3, V5, V7, V13... |

### Variação da taxa de fraude (target)

| Granularidade | Delta máximo | Delta médio |
| --- | --- | --- |
| Mensal | 1.4 p.p. | 0.3 p.p. |
| Trimestral | 0.8 p.p. | 0.5 p.p. |
| Semestral | 0.9 p.p. | 0.9 p.p. |

### Variáveis de maior risco (KS máximo)

1. **id_13** — KS=0.435 (distribuição quase irreconhecível entre meses)
2. **addr1_tx_count** — KS=0.406 (feature comportamental muda rapidamente)
3. **id_17** — KS=0.280
4. **card1_tx_count** — KS=0.191
5. **id_02** — KS=0.149

### Recomendação: retreino mensal com gatilho de monitoramento semanal

**Cadência base: mensal.**

Justificativa estatística:

* **9% das variáveis (11/120) apresentam drift severo já na granularidade mensal** — id_13 e addr1_tx_count têm PSI > 3.0, indicando que a distribuição muda radicalmente de mês para mês. Retreino trimestral permitiria que essas variáveis acumulassem 3 meses de degradação.
* **A taxa de fraude varia até 1.4 p.p. por mês** — um modelo calibrado para a taxa atual estaria desalinhado em ~3-4 p.p. após um trimestre sem retreino, impactando precision-recall e threshold de decisão.
* **Features comportamentais (tx_count, amt_mean) driftam mais rápido** — addr1_tx_count e card1_tx_count são agregados de comportamento que mudam conforme padrões de ataque evoluem. São as variáveis que mais justificam retreino frequente.
* **100 variáveis estáveis** — a maioria do modelo (V-features, C-features, TransactionAmt) é estável, então retreino semanal seria desnecessário e operacionalmente custoso.
* **Features D (time-delta) driftam moderadamente** — D2, D4, D10, D11 mostram PSI 0.15–0.29 mensal, consistente com sazonalidade temporal. Retreino mensal captura essa variação.

**Gatilho de retreino antecipado (semanal):**

Monitorar semanalmente as 3 variáveis de maior PSI (id_13, addr1_tx_count, card1_tx_count). Se qualquer uma exceder 2x o PSI máximo observado mensalmente, disparar retreino imediato sem aguardar o ciclo mensal.